# 1. Title, team members and project summary
## Project 4 — Machine Learning in Climate Analysis and Prediction
**Team 4 · Introduction to Machine Learning with Python**  
**Team members:** Miguel De Jesus Rosa; Elias (surname pending confirmation).

This project studies historical warming and its association with atmospheric CO₂, forecasts monthly temperatures for ten cities, explores climate groups using PCA and K-Means, and classifies Australian rainfall and local extreme-heat events. It uses observed Berkeley Earth, NOAA, Rain in Australia, and Meteostat records, chronological validation, and fixed random seeds. The team reports successful execution; this uploaded copy contains no saved execution outputs, so quantitative findings and a final results summary must be added from the executed notebook before submission.

The nine numbered main sections below follow D1 of the assignment. See docs/RUBRIC_ALIGNMENT.md in the repository for the remaining submission requirements.


## 2. Imports, configuration and data loading
The following Phase 1 cells are preserved from the uploaded working acquisition notebook. They download the source files into `data/raw` and select ten cities. Run on a fresh Colab runtime with network access.

In [ ]:
%pip -q install pandas numpy matplotlib seaborn kagglehub meteostat

In [ ]:
from pathlib import Path
from datetime import date
import json, shutil, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
FIG = ROOT / 'figures'
OUT = ROOT / 'outputs' / 'phase1'
for folder in (RAW, FIG, OUT): folder.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.figsize': (11, 5), 'axes.grid': True, 'grid.alpha': .25})
SEED = 42
GLOBAL_COL = 'LandAndOceanAverageTemperature'
CITIES = [('New York','United States'),('London','United Kingdom'),('Paris','France'),
          ('Cairo','Egypt'),('Bombay','India'),('Tokyo','Japan'),
          ('Sydney','Australia'),('Cape Town','South Africa'),
          ('São Paulo','Brazil'),('La Plata','Argentina')]
print('Carpeta de trabajo:', ROOT)

### 2.1. Obtener los datos

Las dos tablas de Berkeley Earth vienen del mismo conjunto de Kaggle. NOAA se descarga directamente de su archivo mensual. Meteostat proporciona datos diarios de la estación local. Los archivos quedan almacenados en `data/raw/` y no deben subirse a GitHub si su tamaño es excesivo. Si una descarga falla, revisa la conectividad o copia el archivo correspondiente a esa carpeta y vuelve a ejecutar la celda.

In [ ]:
import kagglehub

def kaggle_file(handle, filename):
    target = RAW / filename
    if not target.exists():
        cached = Path(kagglehub.dataset_download(handle, path=filename))
        shutil.copy2(cached, target)
    return target

paths = {}
handle = 'berkeleyearth/climate-change-earth-surface-temperature-data/versions/2'
paths['global'] = kaggle_file(handle, 'GlobalTemperatures.csv')
paths['cities'] = kaggle_file(handle, 'GlobalLandTemperaturesByCity.csv')
paths['rain'] = kaggle_file('jsphyg/weather-dataset-rattle-package/versions/2', 'weatherAUS.csv')
co2path = RAW / 'co2_mm_mlo.csv'
if not co2path.exists():
    urllib.request.urlretrieve('https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv', co2path)
paths['co2'] = co2path
print({k: str(v) for k,v in paths.items()})

In [ ]:
# Meteostat: estación validada previamente para UAGM Gurabo.
STATION_ID = '78526'
station_path = RAW / 'meteostat_78526_2000-2025.csv'
if not station_path.exists():
    try:
        import meteostat as ms
        station_data = ms.daily(ms.Station(id=STATION_ID), date(2000,1,1), date(2025,12,31),
                                providers=[ms.Provider.DAILY]).fetch()
        if station_data.empty: raise ValueError('La estación no devolvió observaciones')
        station_data.to_csv(station_path, index_label='time')
    except Exception as exc:
        print('No se pudo descargar Meteostat:', repr(exc))
if station_path.exists():
    paths['station'] = station_path
print('Estación disponible:', 'station' in paths)

### 2.2. Cargar datos y revisar las primeras filas

`shape`, `head`, tipos de columna, intervalos temporales y faltantes se guardan en las tablas de auditoría. Las concentraciones inválidas codificadas como números negativos en NOAA se convierten en valores faltantes.

In [ ]:
glob = pd.read_csv(paths['global'], parse_dates=['dt'])
co2 = pd.read_csv(paths['co2'], comment='#', skipinitialspace=True)
co2.columns = co2.columns.str.strip()
co2['dt'] = pd.to_datetime(dict(year=co2['year'],month=co2['month'],day=1))
for col in ['average','deseasonalized','ndays','sdev','unc']:
    if col in co2: co2[col] = co2[col].mask(co2[col] < 0)
rain = pd.read_csv(paths['rain'],parse_dates=['Date'])
station = pd.read_csv(paths['station'],parse_dates=['time']) if 'station' in paths else None
for name, frame in [('Global',glob),('NOAA CO2',co2),('Lluvia Australia',rain),('Estación',station)]:
    if frame is None: continue
    print('\n',name, '| shape=',frame.shape)
    display(frame.head())
    display((frame.isna().mean()*100).sort_values(ascending=False).head(12).round(2).rename('% faltante').to_frame())

### 2.3. Analizar los registros de ciudades por fragmentos

Esta estrategia evita cargar el archivo completo en memoria. Se cuentan todas las filas y las claves geográficas y se retienen los registros de nuestras diez ciudades. Para la selección se utiliza **Bombay** como identificador histórico de Mumbai y **La Plata** en lugar de Buenos Aires, que no aparece en el archivo utilizado.

In [ ]:
selected_parts=[]; city_missing=None; city_count=0; keys=set(); earliest=None; latest=None
keycols=['City','Country','Latitude','Longitude']
for chunk in pd.read_csv(paths['cities'],parse_dates=['dt'],chunksize=250_000):
    city_count += len(chunk)
    miss = chunk.isna().sum()
    city_missing = miss if city_missing is None else city_missing.add(miss,fill_value=0)
    keys.update(map(tuple,chunk[keycols].drop_duplicates().itertuples(index=False,name=None)))
    mi,ma=chunk['dt'].min(),chunk['dt'].max()
    earliest=mi if earliest is None else min(earliest,mi)
    latest=ma if latest is None else max(latest,ma)
    city_pairs=pd.MultiIndex.from_frame(chunk[['City','Country']])
    selected_parts.append(chunk.loc[city_pairs.isin(CITIES)].copy())
selected=pd.concat(selected_parts,ignore_index=True)
matched=set(map(tuple,selected[['City','Country']].drop_duplicates().itertuples(index=False,name=None)))
assert set(CITIES).issubset(matched), f'Faltan ciudades: {set(CITIES)-matched}'
print(f'Ciudades: {city_count:,} registros; {len(keys):,} claves geográficas; {earliest.date()} a {latest.date()}')
print(f'Selección: {len(selected):,} filas para 10 ciudades')

In [ ]:
def audit(name,df,date_col,location_col=None):
    dates=pd.to_datetime(df[date_col])
    return dict(dataset=name, rows=len(df), columns=len(df.columns), first_date=str(dates.min().date()),
                last_date=str(dates.max().date()), locations=int(df[location_col].nunique()) if location_col else 1,
                missing_cells=int(df.isna().sum().sum()))
summary=[audit('Global',glob,'dt'),audit('NOAA CO2',co2,'dt'),audit('Rain Australia',rain,'Date','Location')]
summary.append(dict(dataset='Berkeley cities (full)',rows=city_count,columns=len(city_missing),
                    first_date=str(earliest.date()),last_date=str(latest.date()),locations=len(keys),
                    missing_cells=int(city_missing.sum())))
if station is not None: summary.append(audit('Meteostat 78526',station,'time'))
summary=pd.DataFrame(summary)
display(summary)
summary.to_csv(OUT/'data_coverage.csv',index=False)

records=[]
for (city,country),part in selected.groupby(['City','Country']):
    valid=part.dropna(subset=['AverageTemperature'])
    test=valid[valid['dt'].between('2001-01-01','2013-12-31')]
    records.append(dict(city=city,country=country,rows=len(part),valid_months=len(valid),
                        first_valid=str(valid.dt.min().date()),last_valid=str(valid.dt.max().date()),
                        test_valid_months=len(test),test_expected_months=156))
city_cov=pd.DataFrame(records).sort_values('city')
display(city_cov)
city_cov.to_csv(OUT/'city_coverage.csv',index=False)

## 3. Exploratory data analysis
At least six interpretable figures, observed-data coverage and a 1951–1980 anomaly reference period.

### 3.1. Anomalías respecto al período 1951–1980

Para cada año con doce meses observados, la anomalía se define como:  
\[ A_y = T_y - \overline{T}_{1951:1980} \]  
La media de referencia utiliza **30 promedios anuales completos**. Este enfoque evita dar mayor peso a años con datos mensuales incompletos. Una anomalía positiva significa un año más cálido que el promedio del período de referencia.

In [ ]:
monthly=glob.set_index('dt')[GLOBAL_COL].sort_index().loc['1850':]
annual=monthly.resample('YS').agg(['mean','count'])
annual.loc[annual['count']<12,'mean']=np.nan
base=annual.loc['1951':'1980','mean']
assert base.notna().sum()==30, 'La línea base requiere 30 años con 12 meses completos'
baseline=float(base.mean())
annual['anomaly']=annual['mean']-baseline
annual['rolling_10y']=annual['anomaly'].rolling(10,min_periods=10).mean()
annual.index.name='year'
annual.to_csv(OUT/'global_annual_anomalies_1951_1980.csv')
print(f'Temperatura media de referencia 1951–1980: {baseline:.3f} °C')
display(annual[['mean','count','anomaly']].loc['1951':'1955'])

### 3.2. Siete gráficas de análisis exploratorio

Cada figura se guarda como PNG en `figures/`. Las gráficas son **descriptivas**, no son resultados de modelos entrenados.

In [ ]:
def save(name):
    plt.tight_layout(); plt.savefig(FIG/name,dpi=160,bbox_inches='tight'); plt.show()

# Figura 1: temperatura global anual
ax=annual['mean'].plot(label='Temperatura anual',alpha=.7)
annual['mean'].rolling(10,min_periods=10).mean().plot(ax=ax,label='Promedio móvil 10 años',lw=2.5)
ax.set(title='1. Temperatura global anual (Berkeley Earth)',xlabel='Año',ylabel='Temperatura (°C)'); ax.legend()
save('01_global_temperature.png')

# Figura 2: anomalías
ax=annual['anomaly'].plot(label='Anomalía anual',alpha=.6)
annual['rolling_10y'].plot(ax=ax,label='Media móvil 10 años',lw=2.5)
ax.axhline(0,color='black',lw=.9)
ax.set(title='2. Anomalías globales respecto a 1951–1980',xlabel='Año',ylabel='Anomalía (°C)'); ax.legend()
save('02_global_anomalies.png')

# Figura 3: CO2
ax=co2.set_index('dt')['average'].plot()
ax.set(title='3. CO₂ mensual, Mauna Loa (NOAA)',xlabel='Año',ylabel='CO₂ (ppm)')
save('03_co2.png')

In [ ]:
# Figura 4: ciclos mensuales de 3 ciudades con climas contrastantes
subset=selected[selected['City'].isin(['London','Cairo','Sydney'])].dropna(subset=['AverageTemperature']).copy()
subset['month']=subset.dt.dt.month
cycles=subset.groupby(['City','month'])['AverageTemperature'].mean().unstack(0)
ax=cycles.plot(marker='o')
ax.set(title='4. Ciclo estacional de temperatura: Londres, El Cairo y Sídney',xlabel='Mes',ylabel='Temperatura media histórica (°C)')
ax.set_xticks(range(1,13));save('04_monthly_cycles.png')

# Figura 5: frecuencia de lluvia por ubicación australiana
rain_valid=rain.dropna(subset=['RainTomorrow'])
frequency=rain_valid.assign(rain_yes=rain_valid['RainTomorrow'].eq('Yes')).groupby('Location')['rain_yes'].mean().sort_values()
fig,ax=plt.subplots(figsize=(13,10))
frequency.plot.barh(ax=ax)
ax.set(title='5. Proporción observada de lluvia al día siguiente por ubicación australiana',xlabel='Proporción de días con lluvia',ylabel='Ubicación')
save('05_rain_frequency.png')

In [ ]:
# Figura 6: faltantes por variable en datos de Australia
frac=rain.isna().mean().sort_values(ascending=False)
ax=(100*frac).plot.bar(figsize=(12,5))
ax.set(title='6. Porcentaje de valores faltantes — Rain Australia',xlabel='Variable',ylabel='Faltantes (%)')
save('06_missingness.png')

# Figura 7: incertidumbre reportada en temperatura global
unc='LandAndOceanAverageTemperatureUncertainty'
if unc not in glob.columns: raise KeyError(f'Columna de incertidumbre no encontrada: {unc}')
uncertainty=glob.set_index('dt')[unc].resample('YS').mean().loc['1850':]
ax=uncertainty.plot()
ax.set(title='7. Incertidumbre media anual en el registro global',xlabel='Año',ylabel='Incertidumbre informada (°C)')
save('07_source_uncertainty.png')

### 3.3. Verificación y conclusiones de Fase 1

**Interpretación guiada:**

1. ¿La temperatura global aumenta en las décadas más recientes? Comparar datos anuales con media móvil.
2. ¿Qué significa una anomalía positiva o negativa respecto de 1951–1980?
3. ¿Cómo cambia el CO₂ a lo largo del registro disponible?
4. ¿Por qué Londres, El Cairo y Sídney presentan distintos ciclos estacionales?
5. ¿Qué ubicaciones de Australia muestran mayor frecuencia histórica de lluvia al día siguiente?
6. ¿Qué columnas tienen más valores faltantes? ¿Cómo podrían afectar las predicciones?
7. ¿La incertidumbre es mayor en los primeros años del registro?

**Limitaciones:** La temperatura de Berkeley Earth no tiene las mismas variables que el conjunto australiano; no se pueden aplicar directamente los predictores australianos a las diez ciudades globales. También hay diferencias en períodos de cobertura y valores faltantes. Estas limitaciones deberán aparecer en el reporte IEEE.

In [ ]:
assert len(CITIES)==10 and len(city_cov)==10
assert int(base.notna().sum())==30
assert len(list(FIG.glob('0[1-7]_*.png')))>=7
assert (OUT/'data_coverage.csv').exists()
print('FASE 1: verificaciones básicas aprobadas')
print('Figuras generadas:')
for file in sorted(FIG.glob('0[1-7]_*.png')): print(' -',file)
print('Archivos de auditoría:', list(OUT.glob('*.csv')))

## 4. Preprocessing and train / validation / test split
Time order is preserved in every experiment. Model-specific transformations are fit **inside** training pipelines; never on validation or test partitions. Feature windows for time-series prediction are constructed from observations available at the forecast origin.
City regression: training through 2000, test 2001–2013. Rain: training through 2015, test from 2016. Local heat: final five calendar years held out. Temporal CV and preprocessing pipelines are defined in the model cells. Review duplicate-date boundaries in rain CV and preserve the untouched test set for final evaluation.


In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVR, SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
import time
MODEL_RESULTS=[]
REGRESSION_ROWS=[]
CLASSIFICATION_ROWS=[]
SEED=42

def save_fig(name):
    plt.tight_layout();plt.savefig(FIG/name,dpi=150,bbox_inches='tight');plt.show()


## 5. Unsupervised analysis (PCA and k-Means)
Use 1961–1990 normals for the ten selected cities; for the required **20-city Köppen comparison**, add ten further well-documented cities from the Berkeley source. Do not equate K-Means cluster IDs to Köppen climate classes.

In [ ]:
# Additional well-known cities are selected only for the Phase 4 reference comparison.
EXTRA_CITIES=[('Los Angeles','United States'),('Chicago','United States'),('Berlin','Germany'),
 ('Moscow','Russia'),('Beijing','China'),('Bangkok','Thailand'),('Singapore','Singapore'),
 ('Melbourne','Australia'),('Lima','Peru'),('Nairobi','Kenya')]
all_pairs=CITIES+EXTRA_CITIES
parts=[selected]
for chunk in pd.read_csv(paths['cities'],parse_dates=['dt'],chunksize=250_000):
    mask=pd.MultiIndex.from_frame(chunk[['City','Country']]).isin(EXTRA_CITIES)
    sub=chunk.loc[mask]
    if not sub.empty:parts.append(sub)
climate_cities=pd.concat(parts,ignore_index=True)
climate_cities['month']=climate_cities.dt.dt.month
normals=climate_cities[climate_cities.dt.dt.year.between(1961,1990)]
clim=normals.groupby(['City','Country','month'])['AverageTemperature'].agg(['mean','count']).reset_index()
clim.loc[clim['count']<20,'mean']=np.nan
wide=clim.pivot(index=['City','Country'],columns='month',values='mean').reindex(columns=range(1,13)).dropna()
geo=normals.groupby(['City','Country'])[['Latitude','Longitude']].first().reindex(wide.index)
def parse_degree(s):
    if isinstance(s,(int,float)):return float(s)
    v=str(s).strip();return float(v[:-1])*(-1 if v[-1] in 'SW' else 1)
geo=geo.map(parse_degree)
profile=wide.copy();profile['annual_range']=wide.max(axis=1)-wide.min(axis=1);profile['latitude']=geo.Latitude
if len(profile)<10:raise RuntimeError(f'Not enough complete reference cities: {len(profile)}')
X=StandardScaler().fit_transform(profile)
coords=PCA(n_components=2).fit_transform(X)
components=PCA(n_components=2).fit(X)
fig,ax=plt.subplots(figsize=(11,7));ax.scatter(coords[:,0],coords[:,1])
for i,(city,country) in enumerate(profile.index):ax.annotate(city,coords[i],fontsize=8)
ax.set(xlabel='PC1',ylabel='PC2',title=f'Climate PCA · variance {components.explained_variance_ratio_.round(3)}')
save_fig('phase4_pca.png')
display(pd.DataFrame(components.components_.T,index=profile.columns,columns=['PC1','PC2']))
ks=range(2,min(10,len(profile)-1)+1);diag=[]
for k in ks:
    km=KMeans(n_clusters=k,n_init=20,random_state=SEED).fit(X)
    diag.append({'k':k,'inertia':km.inertia_,'silhouette':silhouette_score(X,km.labels_)})
clust_diag=pd.DataFrame(diag);display(clust_diag)
fig,axs=plt.subplots(1,2,figsize=(12,4));axs[0].plot(clust_diag.k,clust_diag.inertia,'o-');axs[0].set_title('Elbow');axs[1].plot(clust_diag.k,clust_diag.silhouette,'o-');axs[1].set_title('Silhouette');save_fig('phase4_cluster_diagnostics.png')
best_k=int(clust_diag.loc[clust_diag.silhouette.idxmax(),'k'])
labels=KMeans(n_clusters=best_k,n_init=20,random_state=SEED).fit_predict(X)
clusters=geo.copy();clusters['cluster']=labels;clusters['city']=[i[0] for i in clusters.index]
fig,ax=plt.subplots(figsize=(13,6));ax.scatter(clusters.Longitude,clusters.Latitude,c=labels,cmap='tab10',s=80)
for r in clusters.itertuples():ax.annotate(r.city,(r.Longitude,r.Latitude),fontsize=7)
ax.set(xlim=(-180,180),ylim=(-90,90),xlabel='Longitude',ylabel='Latitude',title='Geographical locations of K-Means climate clusters')
save_fig('phase4_geo_clusters.png')
print('K selected:',best_k,'Cities:',len(profile))
# Köppen mapping must come from cited, independently verified classifications.
koppen_path=RAW/'koppen_reference.csv'
if koppen_path.exists():
    reference=pd.read_csv(koppen_path)
    assert {'City','Country','Koppen'}.issubset(reference.columns)
    check=clusters.reset_index(drop=True).merge(reference,on=['City','Country']) if 'City' in clusters.columns else clusters.reset_index().merge(reference,on=['City','Country'])
    if len(check)<20:print('Köppen comparison needs at least 20 matched reference cities:',len(check))
    else:
        display(pd.crosstab(check.cluster,check.Koppen));print('Adjusted Rand:',adjusted_rand_score(check.Koppen,check.cluster))
else:print('Köppen comparison PENDING: provide sourced data/raw/koppen_reference.csv with City,Country,Koppen (20 cities).')


## 6. Supervised models
Each model is trained only on the training period, with hyperparameters selected through cross-validation. The test interval is untouched during tuning.

### 6.1. Global warming trends and atmospheric CO₂ — Phase 2


In [ ]:
from scipy.stats import linregress, t
trend=annual.dropna(subset=['anomaly']).copy()
trend['year']=trend.index.year
for label,frame in [('1850–1974',trend[trend.year<=1974]),('1975–latest',trend[trend.year>=1975]),('All',trend)]:
    lr=linregress(frame.year,frame.anomaly)
    critical=t.ppf(.975,len(frame)-2)
    lo,hi=(lr.slope-critical*lr.stderr)*10,(lr.slope+critical*lr.stderr)*10
    print(label,'warming °C/decade:',round(lr.slope*10,4),'95% CI:',(round(lo,4),round(hi,4)))
    MODEL_RESULTS.append({'task':'global trend','model':label,'test_metric':'warming_C_per_decade','value':lr.slope*10})
from sklearn.metrics import mean_squared_error
for degree in (1,2,3):
    x=trend[['year']];y=trend.anomaly
    pipeline=Pipeline([('scale',StandardScaler()),('poly',PolynomialFeatures(degree=degree,include_bias=False)),('model',LinearRegression())])
    pipeline.fit(x,y)
    print('Descriptive trend degree',degree,'RMSE:',round(np.sqrt(mean_squared_error(y,pipeline.predict(x))),4))
co2annual=co2.assign(year=co2.dt.dt.year).groupby('year')['average'].mean().rename('co2')
joined=trend.set_index('year')[['anomaly']].join(co2annual).dropna()
joined=joined.loc[1958:]
for with_co2 in (False,True):
    columns=['year']+(['co2'] if with_co2 else [])
    frame=joined.reset_index();X=frame[columns];y=frame.anomaly
    for model in (Ridge(),Lasso(max_iter=20000)):
        pipe=Pipeline([('scale',StandardScaler()),('reg',model)])
        grid={'reg__alpha':[0.001,0.01,0.1,1,10,100]}
        split=TimeSeriesSplit(n_splits=4)
        fit=GridSearchCV(pipe,grid,cv=split,scoring='neg_mean_squared_error')
        fit.fit(X,y)
        print(model.__class__.__name__, 'CO2:',with_co2,'best:',fit.best_params_,'CV RMSE:',round(np.sqrt(-fit.best_score_),3))


### 6.2. Monthly temperature regression for ten cities — Phase 3
The implemented lagged experiment forecasts one month ahead. A separate leakage-safe 12-month-ahead experiment remains required by the assignment research question; it is not implemented in this copy.


In [ ]:
def forecasting_features(city):
    g=city.sort_values('dt').drop_duplicates('dt').set_index('dt').asfreq('MS')
    y=g.AverageTemperature
    f=pd.DataFrame({'target':y,'year':g.index.year,'sin_month':np.sin(2*np.pi*g.index.month/12),'cos_month':np.cos(2*np.pi*g.index.month/12)},index=g.index)
    for lag in [1,2,3,12]:f['lag_'+str(lag)]=y.shift(lag)
    f['rolling_12']=y.shift(1).rolling(12,min_periods=12).mean()
    return f.dropna()
reg_models={
 'Linear':(LinearRegression(),{}),
 'Ridge':(Ridge(),{'model__alpha':[.1,1,10]}),
 'Lasso':(Lasso(max_iter=10000),{'model__alpha':[.001,.01,.1]}),
 'Polynomial':(Ridge(),{'poly__degree':[2],'model__alpha':[1,10]}),
 'kNN':(KNeighborsRegressor(),{'model__n_neighbors':[3,7]}),
 'Decision tree':(DecisionTreeRegressor(random_state=SEED),{'model__max_depth':[4,8]}),
 'Random forest':(RandomForestRegressor(n_estimators=80,n_jobs=-1,random_state=SEED),{'model__max_depth':[5,None]}),
 'Gradient boosting':(GradientBoostingRegressor(random_state=SEED),{'model__n_estimators':[80,120]}),
 'SVR':(SVR(),{'model__C':[1,10]}),
 'MLP':(MLPRegressor(hidden_layer_sizes=(32,),max_iter=800,random_state=SEED,early_stopping=True),{'model__alpha':[.001,.1]})}
# Change to False for quicker Colab debugging; final notebook should retain True.
RUN_FULL_MODELS=True
regression_predictions={}
for (city,country),group in selected.groupby(['City','Country']):
    f=forecasting_features(group)
    train=f.loc[:'2000-12-01'];test=f.loc['2001-01-01':'2013-12-01']
    if len(train)<60 or len(test)<100:print('Skipping insufficient coverage:',city);continue
    Xtr=train.drop(columns='target');ytr=train.target;Xte=test.drop(columns='target');yte=test.target
    cv=TimeSeriesSplit(n_splits=3)
    baseline=test.lag_12
    REGRESSION_ROWS.append(dict(city=city,model='Seasonal naive',params='-',cv_mean=np.nan,cv_sd=np.nan,MAE=mean_absolute_error(yte,baseline),RMSE=np.sqrt(mean_squared_error(yte,baseline)),R2=r2_score(yte,baseline),train_seconds=0,predict_seconds=0))
    regression_predictions[city]=pd.DataFrame({'actual':yte,'seasonal_naive':baseline},index=test.index)
    for name,(estimator,grid) in reg_models.items():
        steps=[('scale',StandardScaler())]
        if name=='Polynomial':steps.append(('poly',PolynomialFeatures(include_bias=False)))
        steps.append(('model',estimator));pipe=Pipeline(steps)
        search=GridSearchCV(pipe,grid or {},cv=cv,scoring='neg_mean_absolute_error',n_jobs=-1)
        tic=time.perf_counter();search.fit(Xtr,ytr);train_s=time.perf_counter()-tic
        tic=time.perf_counter();pred=search.predict(Xte);pred_s=time.perf_counter()-tic
        REGRESSION_ROWS.append(dict(city=city,model=name,params=str(search.best_params_),cv_mean=-search.best_score_,cv_sd=search.cv_results_['std_test_score'][search.best_index_],MAE=mean_absolute_error(yte,pred),RMSE=np.sqrt(mean_squared_error(yte,pred)),R2=r2_score(yte,pred),train_seconds=train_s,predict_seconds=pred_s))
        regression_predictions[city][name]=pred
regression_table=pd.DataFrame(REGRESSION_ROWS)
display(regression_table)
regression_table.to_csv(OUT/'phase3_regression_comparison.csv',index=False)
for city in list(regression_predictions)[:3]:
    f=regression_predictions[city]
    best=regression_table[(regression_table.city==city)&(regression_table.model!='Seasonal naive')].sort_values('MAE').iloc[0]['model']
    f[['actual','seasonal_naive',best]].iloc[:60].plot(figsize=(11,4),title=f'{city} | held-out monthly temperatures (°C)')
    save_fig('phase3_forecast_'+city.replace(' ','_')+'.png')


### 6.3. Rain in Australia — Phase 5


In [ ]:
rainframe=rain.copy()
rainframe=rainframe[rainframe.RainTomorrow.isin(['Yes','No'])].copy()
rainframe['label']=(rainframe.RainTomorrow=='Yes').astype(int)
# RainTomorrow is not used as a feature. Rainfall today is valid, while any future-proxy fields must be excluded.
excluded={'RainTomorrow','label','Date','RISK_MM','RainfallTomorrow'}
columns=[c for c in rainframe if c not in excluded]
train=rainframe[rainframe.Date.dt.year<=2015].dropna(subset=['label'])
test=rainframe[rainframe.Date.dt.year>=2016].dropna(subset=['label'])
assert not train.empty and not test.empty
numerical=train[columns].select_dtypes(include=np.number).columns.tolist()
categorical=[c for c in columns if c not in numerical]
pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numerical),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categorical)])
classification_models={
 'Logistic':(LogisticRegression(max_iter=1200,class_weight='balanced'),{'model__C':[.1,1]}),
 'kNN':(KNeighborsClassifier(),{'model__n_neighbors':[9,25]}),
 'Naive Bayes':(GaussianNB(),{}),
 'SVM':(SVC(class_weight='balanced',probability=True),{'model__C':[.1,1]}),
 'Decision tree':(DecisionTreeClassifier(random_state=SEED,class_weight='balanced'),{'model__max_depth':[5,10]}),
 'Random forest':(RandomForestClassifier(n_estimators=100,random_state=SEED,class_weight='balanced',n_jobs=-1),{'model__max_depth':[8,None]}),
 'Gradient boosting':(GradientBoostingClassifier(random_state=SEED),{'model__n_estimators':[75,125]}),
 'MLP':(MLPClassifier(hidden_layer_sizes=(32,),max_iter=300,random_state=SEED),{'model__alpha':[.001,.1]})}
def class_run(train,test,feature_columns,transformer,task,cv):
    out=[];roc=[]
    for name,(model,grid) in classification_models.items():
        pipe=Pipeline([('prep',transformer),('model',model)])
        fit=GridSearchCV(pipe,grid or {},cv=cv,scoring='f1',n_jobs=-1)
        start=time.perf_counter();fit.fit(train[feature_columns],train.label);train_s=time.perf_counter()-start
        start=time.perf_counter();pred=fit.predict(test[feature_columns]);pred_s=time.perf_counter()-start
        prob=fit.predict_proba(test[feature_columns])[:,1]
        out.append(dict(task=task,model=name,params=str(fit.best_params_),cv_mean=fit.best_score_,cv_sd=fit.cv_results_['std_test_score'][fit.best_index_],precision=precision_score(test.label,pred,zero_division=0),recall=recall_score(test.label,pred,zero_division=0),F1=f1_score(test.label,pred,zero_division=0),ROC_AUC=roc_auc_score(test.label,prob),train_seconds=train_s,predict_seconds=pred_s))
        roc.append((name,prob))
    return pd.DataFrame(out),roc
# SVM on full 145k rows can be computationally expensive; deterministic chronological subsampling for tuning.
# Final holdout remains complete and untouched.
cap=24000
train_sample=train.sort_values('Date')
if len(train_sample)>cap:
    # Use uniform temporal sampling without using test labels or future observations.
    ids=np.linspace(0,len(train_sample)-1,cap,dtype=int);train_sample=train_sample.iloc[ids]
print('Training sample',len(train_sample),'of',len(train),'held-out',len(test))
rain_scores,rain_roc=class_run(train_sample,test,columns,pre,'Australia rain',TimeSeriesSplit(n_splits=3))
display(rain_scores);rain_scores.to_csv(OUT/'phase5_rain_comparison.csv',index=False)
fig,ax=plt.subplots(figsize=(8,6))
from sklearn.metrics import roc_curve
for name,p in rain_roc:
    fpr,tpr,_=roc_curve(test.label,p);ax.plot(fpr,tpr,label=name)
ax.plot([0,1],[0,1],'k--');ax.legend(fontsize=8);ax.set(xlabel='False positive rate',ylabel='True positive rate',title='Rain in Australia · held-out ROC')
save_fig('phase5_rain_roc.png')


### 6.4. Local extreme-heat prediction and annual trend — Phase 5


In [ ]:
if station is None or station.empty:
    print('Local station data unavailable; download a verified 20-year daily record and rerun Phase 1.')
else:
    daily=station.copy().set_index('time').sort_index().asfreq('D')
    daily['tmax']=pd.to_numeric(daily['tmax'],errors='coerce')
    if (daily.index.max()-daily.index.min()).days<365*20:raise ValueError('At least 20 years of local daily data required')
    heat_threshold=daily.tmax.quantile(.90)
    daily['label']=(daily.tmax>heat_threshold).astype(float);daily.loc[daily.tmax.isna(),'label']=np.nan
    for lag in [1,2,3,7]:daily[f'tmax_lag{lag}']=daily.tmax.shift(lag)
    daily['tmax_roll7']=daily.tmax.shift(1).rolling(7).mean()
    for var in ['tmin','prcp']:
        if var in daily:daily[var+'_lag1']=daily[var].shift(1)
    daily['month_sin']=np.sin(2*np.pi*daily.index.month/12);daily['month_cos']=np.cos(2*np.pi*daily.index.month/12)
    features=[x for x in daily if x.startswith(('tmax_lag','tmax_roll','tmin_lag','prcp_lag','month_'))]
    heat=daily[features+['label']].reset_index().rename(columns={'time':'Date'}).dropna(subset=['label'])
    cutoff=int(heat.Date.dt.year.max()-5)
    ht=heat[heat.Date.dt.year<=cutoff];hv=heat[heat.Date.dt.year>cutoff]
    local_transform=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),features)])
    heat_scores,heat_roc=class_run(ht,hv,features,local_transform,'Local heat',TimeSeriesSplit(n_splits=4))
    display(heat_scores);heat_scores.to_csv(OUT/'phase5_heat_comparison.csv',index=False)
    fig,ax=plt.subplots(figsize=(8,5))
    for name,p in heat_roc:
        fpr,tpr,_=roc_curve(hv.label,p);ax.plot(fpr,tpr,label=name)
    ax.plot([0,1],[0,1],'k--');ax.legend(fontsize=8);ax.set(xlabel='False positive rate',ylabel='True positive rate',title='Local heat · held-out ROC');save_fig('phase5_heat_roc.png')
    annual_heat=daily.groupby(daily.index.year).agg(heat_days=('label','sum'),observed=('label','count'))
    annual_heat=annual_heat[annual_heat.observed>=300]
    slope,intercept=np.polyfit(annual_heat.index,annual_heat.heat_days,1)
    ax=annual_heat.heat_days.plot(figsize=(10,4),marker='o',label='Observed heat days')
    ax.plot(annual_heat.index,slope*annual_heat.index+intercept,label='Linear trend');ax.legend()
    ax.set(xlabel='Year',ylabel='Days above full-record 90th percentile',title=f'Local heat days (slope {slope:+.2f} days/year)');save_fig('phase5_heat_trend.png')
    print('90th percentile of tmax (°C):',heat_threshold,'trend heat days/year:',slope)


## 7. Model comparison table and plots
Tables display best hyperparameters, cross-validation mean/SD, held-out metrics, and training/prediction times. All metrics below are computed from observed data after successful training. For Phase 5, per-location accuracy and confusion matrices are additionally required by the assignment.

In [ ]:
if REGRESSION_ROWS:
    display(regression_table.groupby('model')[['MAE','RMSE','R2']].mean().sort_values('MAE'))
if 'rain_scores' in globals():
    display(rain_scores.sort_values('F1',ascending=False))
    # Confusion matrix and per-location accuracy for the best F1 model are fit on the same train split.
    best_name=rain_scores.sort_values('F1',ascending=False).iloc[0]['model']
    model,grid=classification_models[best_name]
    search=GridSearchCV(Pipeline([('prep',pre),('model',model)]),grid or {},cv=TimeSeriesSplit(3),scoring='f1',n_jobs=-1)
    search.fit(train_sample[columns],train_sample.label)
    predictions=search.predict(test[columns])
    print('Rain confusion matrix:');display(pd.DataFrame(confusion_matrix(test.label,predictions),index=['Actual No','Actual Yes'],columns=['Pred No','Pred Yes']))
    by_location=test[['Location','label']].copy();by_location['correct']=(predictions==test.label.to_numpy())
    display(by_location.groupby('Location').correct.agg(['mean','count']).sort_values('mean'))
if 'heat_scores' in globals():display(heat_scores.sort_values('F1',ascending=False))


## 8. Interpretation and error analysis
After running all cells: interpret the dominant PCA loadings (warmth/seasonality), the Köppen agreement, model-versus-seasonal-naive skill, per-location rain performance, and whether observed heat days are increasing. Forecast errors and test scores must be discussed with the actual values shown in Section 7. A high CO₂–temperature correlation does **not** independently prove causation.

In [ ]:
if REGRESSION_ROWS:
    grp=regression_table.groupby('model').MAE.mean().sort_values();print('Lowest mean monthly-test MAE:',grp.index[0],round(grp.iloc[0],3),'°C')
if 'rain_scores' in globals():
    top=rain_scores.sort_values('F1',ascending=False).iloc[0]
    print(f'Best rain F1 model: {top.model} | F1={top.F1:.3f} | ROC-AUC={top.ROC_AUC:.3f}')
if 'heat_scores' in globals():
    top=heat_scores.sort_values('F1',ascending=False).iloc[0]
    print(f'Best heat F1 model: {top.model} | F1={top.F1:.3f} | ROC-AUC={top.ROC_AUC:.3f}')


## 9. Conclusions and limitations
**Conclusions must reflect the executed results above.** This workflow examines historic anomalies, change in warming slopes, city-level monthly forecasting, geographical climate clustering, and next-day event classification.

**Limitations:** Berkeley Earth early temperatures have sparse coverage and changing spatial sampling; a ten-city group cannot represent every climate regime; local Meteostat records may have gaps or station changes; the full-record extreme-heat definition is descriptive and should be replaced by a training-only reference percentile for operational forecasting; SVM sampling may affect model comparisons; climate cluster IDs are not Köppen classifications. Observational CO₂ associations alone do not establish causal attribution.

**Reproducibility:** Retain downloaded data locally under `data/raw`, pin dependencies, cite original data sources, and execute **Runtime → Restart session and run all** in a fresh Colab runtime. Verify no failures and preserve the generated tables and figures before submitting. The full project also requires a written IEEE paper and presentation.